In [1]:
import pandas as pd
import numpy as np

# =========================
# Data-driven grouping: near / mid / far from WPT.csv
# Idea:
#   1) Compute mean efficiency per type (A..K)
#   2) Sort types by mean efficiency (low -> high)
#   3) Split into 3 contiguous groups that minimize within-group SSE (1D optimal segmentation)
# =========================

PATH = r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\WPT.csv"

# Load + rename columns to English (works even if already English)
df = pd.read_csv(PATH).rename(columns={"種類": "type", "負載": "load", "效率": "efficiency"})
df["type"] = df["type"].astype(str)
df["load"] = pd.to_numeric(df["load"], errors="coerce")
df["efficiency"] = pd.to_numeric(df["efficiency"], errors="coerce")
df = df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

# 1) Mean efficiency per type
type_stats = (
    df.groupby("type")["efficiency"]
      .agg(n="count", mean="mean", std="std")
      .sort_values("mean")
      .reset_index()
)

print("=== Mean efficiency per type (sorted) ===")
print(type_stats)

# Prepare 1D array (sorted means)
types = type_stats["type"].tolist()
x = type_stats["mean"].to_numpy()
n = len(x)

# Helper: SSE of segment i..j (inclusive) in O(1) using prefix sums
S1 = np.concatenate([[0.0], np.cumsum(x)])
S2 = np.concatenate([[0.0], np.cumsum(x * x)])

def seg_sse(i, j):
    """Sum of squared errors for x[i..j] around their mean."""
    m = j - i + 1
    sum1 = S1[j + 1] - S1[i]
    sum2 = S2[j + 1] - S2[i]
    mu = sum1 / m
    return sum2 - 2 * mu * sum1 + m * mu * mu

# 2) DP for optimal 3-segmentation (contiguous groups)
K = 3  # near, mid, far
INF = 1e18
dp = np.full((K + 1, n + 1), INF)   # dp[k][t] = best cost splitting first t points into k groups
cut = np.full((K + 1, n + 1), -1, dtype=int)

dp[0, 0] = 0.0

# Require each group has at least MIN_SIZE elements (to avoid tiny groups)
MIN_SIZE = 2

for k in range(1, K + 1):
    for t in range(1, n + 1):
        # last group is from p..t-1, so p ranges
        # ensure each group has >= MIN_SIZE
        p_min = (k - 1) * MIN_SIZE
        p_max = t - MIN_SIZE
        if p_max < p_min:
            continue
        best_cost = INF
        best_p = -1
        for p in range(p_min, p_max + 1):
            cost = dp[k - 1, p] + seg_sse(p, t - 1)
            if cost < best_cost:
                best_cost = cost
                best_p = p
        dp[k, t] = best_cost
        cut[k, t] = best_p

if np.isinf(dp[K, n]):
    raise RuntimeError(
        "No valid split found. Try reducing MIN_SIZE to 1 or check number of types."
    )

# Reconstruct segments
bounds = []
t = n
for k in range(K, 0, -1):
    p = cut[k, t]
    bounds.append((p, t))  # [p, t)
    t = p
bounds.reverse()

# 3) Build groups
group_names = ["near", "mid", "far"]
groups = {}
for name, (a, b) in zip(group_names, bounds):
    groups[name] = types[a:b]

print("\n=== Data-driven near/mid/far groups (contiguous by mean efficiency) ===")
for gname in group_names:
    print(f"{gname}: {groups[gname]}")

# Optional: show group summary
print("\n=== Group summary (mean of means / range) ===")
for gname in group_names:
    idx = [types.index(t) for t in groups[gname]]
    vals = x[idx]
    print(
        f"{gname}: size={len(vals)}, mean={vals.mean():.2f}, "
        f"min={vals.min():.2f}, max={vals.max():.2f}"
    )

# Optional: attach the group label back to the dataframe (useful for stage-1 model)
type_to_group = {t: g for g, ts in groups.items() for t in ts}
df["group"] = df["type"].map(type_to_group)

# Save artifacts (optional)
type_stats.to_csv("wpt_type_stats_sorted.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(
    [(t, type_to_group[t]) for t in sorted(type_to_group.keys())],
    columns=["type", "group"]
).to_csv("wpt_type_to_group.csv", index=False, encoding="utf-8-sig")

print("\nSaved: wpt_type_stats_sorted.csv, wpt_type_to_group.csv")

=== Mean efficiency per type (sorted) ===
   type    n       mean        std
0     A   26  59.180000  10.871463
1     C   26  68.575000   9.188302
2     B  104  75.095288   7.748468
3     D  104  76.547115   7.315159
4     E  104  77.470577   7.034445
5     F   26  77.714615   7.748113
6     G  104  78.992788   7.021536
7     I   26  79.641923   7.111834
8     J  208  81.156058   6.414428
9     H  312  81.371282   6.183933
10    K  416  81.829135   6.148433

=== Data-driven near/mid/far groups (contiguous by mean efficiency) ===
near: ['A', 'C']
mid: ['B', 'D', 'E', 'F']
far: ['G', 'I', 'J', 'H', 'K']

=== Group summary (mean of means / range) ===
near: size=2, mean=63.88, min=59.18, max=68.58
mid: size=4, mean=76.71, min=75.10, max=77.71
far: size=5, mean=80.60, min=78.99, max=81.83

Saved: wpt_type_stats_sorted.csv, wpt_type_to_group.csv
